# H-RSSM Experiments V2 — Action Prior, Dimensional Efficiency, OOD Generalisation
**Kaggle GPU notebook** — Experiments 7, 8, 9.

| Exp | Question |
|-----|----------|
| 7 | Does an action-conditioned prior prevent depth-signal decay at 50k steps? |
| 8 | How much hyperbolic geometry is *needed*? H^2 vs R^32 dimensional efficiency. |
| 9 | Does the model generalise to unseen depths (OOD)? |

Results are written to `article_snippets_v2.txt` for copy-paste into the article.


In [ ]:
import subprocess, sys, os, json, re, time, warnings
warnings.filterwarnings("ignore")

# ── Install a GPU-compatible PyTorch ──────────────────────────────────────
# Kaggle T4 = sm_75; torch 2.10+ may drop sm_75 support.
# torch 2.4.1+cu121 covers sm_60–sm_90 and is a stable release.
# --no-deps: swap ONLY the torch binary; leave numpy/scipy untouched.
print("Installing torch 2.4.1+cu121 (--no-deps, ~5 min) …")
_r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--force-reinstall", "--no-deps",
     "torch==2.4.1",
     "--index-url", "https://download.pytorch.org/whl/cu121"],
    capture_output=True, text=True,
)
if _r.returncode != 0:
    print("pip install failed — continuing with existing torch.")
    print(_r.stderr[-400:])
else:
    print("torch 2.4.1+cu121 installed OK")

# Re-import torch after reinstall
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from collections import deque
from typing import List, Tuple, Optional, Dict, Any

# ── Verify CUDA actually works ─────────────────────────────────────────────
def _check_cuda():
    if not torch.cuda.is_available():
        return "cpu"
    try:
        a = torch.zeros(4, device="cuda")
        b = torch.ones(4, 4, device="cuda")
        _ = a @ b
        torch.cuda.synchronize()
        return "cuda"
    except Exception as e:
        print(f"CUDA smoke-test failed: {type(e).__name__}; falling back to CPU.")
        return "cpu"

DEVICE = _check_cuda()
print(f"Device : {DEVICE}  |  torch {torch.__version__}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# ── Step scaling: CPU runs use 5% of the GPU step budget ──────────────────
STEP_SCALE = 1.0 if DEVICE == "cuda" else 0.20
print(f"Step scale: {STEP_SCALE}  ({'full GPU' if DEVICE=='cuda' else 'CPU-reduced (20%)'})")


## Library (inline)
### BaryTreeMDP

In [ ]:
class TreeNode:
    __slots__ = ("idx", "depth", "parent", "children", "obs", "branch_id")
    def __init__(self, idx, depth, parent, branch_id, obs):
        self.idx = idx; self.depth = depth; self.parent = parent
        self.children: List[int] = []; self.branch_id = branch_id; self.obs = obs

class BaryTreeMDP:
    def __init__(self, B=4, L=5, obs_dim=64, seed=42):
        self.B = B; self.L = L; self.obs_dim = obs_dim
        self._build(np.random.RandomState(seed))

    def _build(self, rng):
        dep_dim = 16; br_dim = 16
        depth_embs = rng.randn(self.L+1, dep_dim).astype(np.float32)
        W = (rng.randn(self.obs_dim, dep_dim+br_dim)*0.3).astype(np.float32)
        self.nodes: List[TreeNode] = []
        q = deque([(-1, 0, np.zeros(br_dim, dtype=np.float32), 0)])
        while q:
            par, d, bemb, bid = q.popleft()
            idx = len(self.nodes)
            feat = np.concatenate([depth_embs[d], bemb])
            obs  = np.tanh(W @ feat) + rng.randn(self.obs_dim).astype(np.float32)*0.05
            node = TreeNode(idx, d, par, bid, obs)
            self.nodes.append(node)
            if par >= 0:
                self.nodes[par].children.append(idx)
            if d < self.L:
                for b in range(self.B):
                    q.append((idx, d+1, rng.randn(br_dim).astype(np.float32), b))
        self.n_nodes = len(self.nodes)
        self._adj = [
            list(n.children) + ([n.parent] if n.parent >= 0 else [])
            for n in self.nodes
        ]
        self.nodes_by_depth = [[] for _ in range(self.L+1)]
        for n in self.nodes:
            self.nodes_by_depth[n.depth].append(n.idx)

    def sample_trajectory(self, length=10, start_node=None, max_depth=None, rng=None):
        if rng is None:
            rng = np.random.RandomState()
        if start_node is None:
            pool = [n.idx for n in self.nodes if max_depth is None or n.depth <= max_depth]
            curr = int(rng.choice(pool))
        else:
            curr = start_node
        traj = [curr]
        for _ in range(length - 1):
            n = self.nodes[curr]
            if max_depth is not None:
                nbs = [c for c in n.children if self.nodes[c].depth <= max_depth]
                if n.parent >= 0:
                    nbs.append(n.parent)
            else:
                nbs = self._adj[curr]
            curr = int(rng.choice(nbs)) if nbs else curr
            traj.append(curr)
        return traj

    def sample_batch_with_actions(self, batch_size, seq_len, rng=None, max_depth=None):
        if rng is None:
            rng = np.random.RandomState()
        obs_list, dep_list, br_list, act_list = [], [], [], []
        for _ in range(batch_size):
            traj   = self.sample_trajectory(length=seq_len, max_depth=max_depth, rng=rng)
            depths = [self.nodes[i].depth for i in traj]
            obs_list.append(np.stack([self.nodes[i].obs for i in traj]))
            dep_list.append(depths)
            br_list.append([self.nodes[i].branch_id for i in traj])
            acts = [0] + [int(np.sign(depths[t] - depths[t-1])) for t in range(1, len(traj))]
            act_list.append(acts)
        return (
            np.stack(obs_list).astype(np.float32),
            np.array(dep_list,  dtype=np.int32),
            np.array(br_list,   dtype=np.int32),
            np.array(act_list,  dtype=np.int8),
        )

print("BaryTreeMDP OK")


### Distributions

In [ ]:
class HorosphericalGaussian:
    LOG_MGF_MAX = 20.0

    def __init__(self, mu_tau, mu_b, log_sigma_tau, log_sigma_b):
        self.mu_tau    = mu_tau
        self.mu_b      = mu_b
        self.sigma_tau = torch.exp(log_sigma_tau).clamp(min=1e-4)
        self.sigma_b   = torch.exp(log_sigma_b).clamp(min=1e-4)
        self.d_fibre   = mu_b.shape[-1]

    def rsample(self):
        tau = self.mu_tau + self.sigma_tau * torch.randn_like(self.mu_tau)
        b   = self.mu_b   + self.sigma_b   * torch.randn_like(self.mu_b)
        return tau, b

    def kl_divergence(self, other):
        height_kl = (
            torch.log(other.sigma_tau / self.sigma_tau)
            + (self.sigma_tau**2 + (self.mu_tau - other.mu_tau)**2) / (2.0 * other.sigma_tau**2)
            - 0.5
        )
        fibre_kl = self.d_fibre * (
            torch.log(other.sigma_b / self.sigma_b)
            + self.sigma_b**2 / (2.0 * other.sigma_b**2)
            - 0.5
        )
        mu_b_diff_sq = ((self.mu_b - other.mu_b)**2).sum(dim=-1)
        log_mgf = torch.clamp(
            -2.0 * self.mu_tau + 2.0 * self.sigma_tau**2, max=self.LOG_MGF_MAX
        )
        drift = mu_b_diff_sq / (2.0 * other.sigma_b**2) * torch.exp(log_mgf)
        return height_kl + fibre_kl + drift


class EuclideanGaussian:
    def __init__(self, mu, log_sigma):
        self.mu    = mu
        self.sigma = torch.exp(log_sigma).clamp(min=1e-4)

    def rsample(self):
        return self.mu + self.sigma * torch.randn_like(self.mu)

    def kl_divergence(self, other):
        return (
            torch.log(other.sigma / self.sigma)
            + (self.sigma**2 + (self.mu - other.mu)**2) / (2.0 * other.sigma**2)
            - 0.5
        ).sum(-1)

print("Distributions OK")


### World Models

In [ ]:
def _mlp(in_dim, hid_dim, out_dim, layers=2):
    mods = [nn.Linear(in_dim, hid_dim), nn.ELU()]
    for _ in range(layers - 1):
        mods += [nn.Linear(hid_dim, hid_dim), nn.ELU()]
    mods.append(nn.Linear(hid_dim, out_dim))
    return nn.Sequential(*mods)


class HyperbolicWorldModel(nn.Module):
    def __init__(self, obs_dim=64, latent_dim=16, hidden_dim=256, enc_dim=128):
        super().__init__()
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim
        hg_out = 2 + 2 * (latent_dim - 1)
        self.obs_encoder   = _mlp(obs_dim, enc_dim, enc_dim)
        self.prior_net     = _mlp(hidden_dim, hidden_dim, hg_out)
        self.posterior_net = _mlp(hidden_dim + enc_dim, hidden_dim, hg_out)
        self.decoder       = _mlp(latent_dim, hidden_dim, obs_dim)
        self.gru           = nn.GRUCell(latent_dim, hidden_dim)

    def _parse_hg(self, p):
        d = self.latent_dim - 1
        # p[:, 1] not p[:, 1:2] — keep log_sigma_tau as (batch,) not (batch,1)
        # so sigma_tau stays (batch,) and doesn't broadcast-explode against mu_tau
        return HorosphericalGaussian(p[:, 0], p[:, 2:2+d], p[:, 1], p[:, 2+d:2+2*d])

    def forward(self, obs_seq, action_seq=None):
        batch, T, _ = obs_seq.shape
        dev = obs_seq.device
        h = torch.zeros(batch, self.hidden_dim, device=dev)
        recons, kls, mu_taus, mu_bs, sigma_taus = [], [], [], [], []
        for t in range(T):
            enc  = self.obs_encoder(obs_seq[:, t])
            pri  = self._parse_hg(self.prior_net(h))
            post = self._parse_hg(self.posterior_net(torch.cat([h, enc], -1)))
            tau, b = post.rsample()
            z = torch.cat([tau.unsqueeze(-1), b], -1)
            recons.append(self.decoder(z))
            kls.append(post.kl_divergence(pri))
            mu_taus.append(post.mu_tau.detach())
            mu_bs.append(post.mu_b.detach())
            sigma_taus.append(post.sigma_tau.detach())
            h = self.gru(z, h)
        recons = torch.stack(recons, 1)
        kls    = torch.stack(kls,    1)
        info = {
            "mu_tau":    torch.stack(mu_taus,     1),
            "mu_b":      torch.stack(mu_bs,        1),
            "sigma_tau": torch.stack(sigma_taus,   1),
        }
        return recons, kls, info


class EuclideanWorldModel(nn.Module):
    def __init__(self, obs_dim=64, latent_dim=16, hidden_dim=256, enc_dim=128):
        super().__init__()
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim
        self.obs_encoder   = _mlp(obs_dim, enc_dim, enc_dim)
        self.prior_net     = _mlp(hidden_dim, hidden_dim, 2 * latent_dim)
        self.posterior_net = _mlp(hidden_dim + enc_dim, hidden_dim, 2 * latent_dim)
        self.decoder       = _mlp(latent_dim, hidden_dim, obs_dim)
        self.gru           = nn.GRUCell(latent_dim, hidden_dim)

    def _parse_eg(self, p):
        return EuclideanGaussian(p[:, :self.latent_dim], p[:, self.latent_dim:])

    def forward(self, obs_seq, action_seq=None):
        batch, T, _ = obs_seq.shape
        dev = obs_seq.device
        h = torch.zeros(batch, self.hidden_dim, device=dev)
        recons, kls, mus = [], [], []
        for t in range(T):
            enc  = self.obs_encoder(obs_seq[:, t])
            pri  = self._parse_eg(self.prior_net(h))
            post = self._parse_eg(self.posterior_net(torch.cat([h, enc], -1)))
            z    = post.rsample()
            recons.append(self.decoder(z))
            kls.append(post.kl_divergence(pri))
            mus.append(post.mu.detach())
            h = self.gru(z, h)
        recons = torch.stack(recons, 1)
        kls    = torch.stack(kls,    1)
        info   = {"mu": torch.stack(mus, 1)}
        return recons, kls, info


class HyperbolicWorldModelActionPrior(HyperbolicWorldModel):
    def __init__(self, obs_dim=64, latent_dim=16, hidden_dim=256, enc_dim=128):
        super().__init__(obs_dim, latent_dim, hidden_dim, enc_dim)
        self.log_depth_bias   = nn.Parameter(torch.tensor(-1.0))
        self.log_shallow_bias = nn.Parameter(torch.tensor(-1.0))

    def _action_delta(self, at):
        deeper    = (at > 0).float()
        shallower = (at < 0).float()
        return (  deeper    * F.softplus(self.log_depth_bias)
                - shallower * F.softplus(self.log_shallow_bias))

    def forward(self, obs_seq, action_seq=None):
        batch, T, _ = obs_seq.shape
        dev = obs_seq.device
        h = torch.zeros(batch, self.hidden_dim, device=dev)
        recons, kls, mu_taus, mu_bs, sigma_taus = [], [], [], [], []
        for t in range(T):
            enc = self.obs_encoder(obs_seq[:, t])
            pri = self._parse_hg(self.prior_net(h))
            if action_seq is not None:
                pri.mu_tau = pri.mu_tau + self._action_delta(
                    action_seq[:, t].to(dev).float()
                )
            post = self._parse_hg(self.posterior_net(torch.cat([h, enc], -1)))
            tau, b = post.rsample()
            z = torch.cat([tau.unsqueeze(-1), b], -1)
            recons.append(self.decoder(z))
            kls.append(post.kl_divergence(pri))
            mu_taus.append(post.mu_tau.detach())
            mu_bs.append(post.mu_b.detach())
            sigma_taus.append(post.sigma_tau.detach())
            h = self.gru(z, h)
        recons = torch.stack(recons, 1)
        kls    = torch.stack(kls,    1)
        info = {
            "mu_tau":    torch.stack(mu_taus,     1),
            "mu_b":      torch.stack(mu_bs,        1),
            "sigma_tau": torch.stack(sigma_taus,   1),
        }
        return recons, kls, info

print("World models OK")


### Metrics & training

In [ ]:
def elbo_loss(recons, kls, obs_seq, beta=1.0):
    recon_loss = F.mse_loss(recons, obs_seq, reduction="none").sum(-1).mean()
    kl_loss    = kls.mean()
    return recon_loss + beta * kl_loss, recon_loss, kl_loss


def eval_model(model, env, rng, n_batches=20, batch=64, seq=20,
               use_actions=False, max_depth=None):
    model.eval()
    recon_losses, mu_taus, depths = [], [], []
    with torch.no_grad():
        for _ in range(n_batches):
            obs, deps, _, acts = env.sample_batch_with_actions(
                batch, seq, rng=rng, max_depth=max_depth)
            obs_t  = torch.from_numpy(obs).to(DEVICE)
            acts_t = (torch.from_numpy(acts.astype(np.float32)).to(DEVICE)
                      if use_actions else None)
            recons, kls, info = model(obs_t, acts_t)
            loss, rl, kl = elbo_loss(recons, kls, obs_t)
            recon_losses.append(rl.item())
            mu_taus.append(info["mu_tau"].cpu().numpy())
            depths.append(deps)
    mu_all  = np.concatenate(mu_taus, axis=0).reshape(-1)
    dep_all = np.concatenate(depths,  axis=0).reshape(-1)
    rho, _  = spearmanr(mu_all, dep_all)
    return {"recon": float(np.mean(recon_losses)), "rho": float(rho)}


def rho_on_depth_range(model, env, rng, min_d, max_d,
                       n_batches=20, batch=64, seq=20,
                       use_actions=False):
    model.eval()
    mu_list, dep_list = [], []
    with torch.no_grad():
        for _ in range(n_batches):
            obs, deps, _, acts = env.sample_batch_with_actions(batch, seq, rng=rng)
            obs_t  = torch.from_numpy(obs).to(DEVICE)
            acts_t = (torch.from_numpy(acts.astype(np.float32)).to(DEVICE)
                      if use_actions else None)
            _, _, info = model(obs_t, acts_t)
            mu_list.append(info["mu_tau"].cpu().numpy())
            dep_list.append(deps)
    mu_all  = np.concatenate(mu_list,  axis=0).reshape(-1)
    dep_all = np.concatenate(dep_list, axis=0).reshape(-1)
    mask = (dep_all >= min_d) & (dep_all <= max_d)
    if mask.sum() < 10:
        return float("nan")
    rho, _ = spearmanr(mu_all[mask], dep_all[mask])
    return float(rho)


def train(model, env, steps, beta=1.0, lr=3e-4, batch=64, seq=20,
          use_actions=False, max_depth=None, checkpoint_steps=None, rng=None):
    if rng is None:
        rng = np.random.RandomState(0)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    model.train()
    t0 = time.time()
    for step in range(1, steps + 1):
        obs, deps, _, acts = env.sample_batch_with_actions(
            batch, seq, rng=rng, max_depth=max_depth)
        obs_t  = torch.from_numpy(obs).to(DEVICE)
        acts_t = (torch.from_numpy(acts.astype(np.float32)).to(DEVICE)
                  if use_actions else None)
        recons, kls, info = model(obs_t, acts_t)
        loss, rl, kl = elbo_loss(recons, kls, obs_t, beta=beta)
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        if checkpoint_steps and step in checkpoint_steps:
            metrics = eval_model(model, env, rng,
                                 use_actions=use_actions, max_depth=max_depth)
            metrics["step"] = step
            history.append(metrics)
            model.train()
            print(f"  step {step:5d} | rho={metrics['rho']:+.3f} | "
                  f"recon={metrics['recon']:.4f} | {time.time()-t0:.0f}s")
    if not checkpoint_steps:
        metrics = eval_model(model, env, rng,
                             use_actions=use_actions, max_depth=max_depth)
        history.append(metrics)
    return history

print("Helpers OK")


## Experiment 7 — Action-conditioned prior stability (50k steps)

**Hypothesis:** without explicit structural guidance, the ELBO reconstruction term erodes
the depth signal at 50k steps (ρ drops from 0.30 → 0.20). Injecting the action into the
prior as `μ_τ^p += δ(a_t)` — where δ(+1) > 0 and δ(−1) < 0 — makes depth alignment an
explicit objective, counteracting the shallow attractor.


In [ ]:
STEPS_7 = int(50_000 * STEP_SCALE)
N_CKPTS = 10  # always 10 checkpoints regardless of step count
CKPT_INTERVAL = max(1, STEPS_7 // N_CKPTS)
CKPTS_7 = list(range(CKPT_INTERVAL, STEPS_7 + 1, CKPT_INTERVAL))
print(f"Exp 7: {STEPS_7} steps, checkpoints every {CKPT_INTERVAL}")
env7    = BaryTreeMDP(B=4, L=5, obs_dim=64, seed=42)

print("=== Standard HG (no action prior) ===")
rng7_std  = np.random.RandomState(7)
std7      = HyperbolicWorldModel(obs_dim=64, latent_dim=16).to(DEVICE)
hist_std7 = train(std7, env7, steps=STEPS_7, beta=1.0,
                  use_actions=False, checkpoint_steps=CKPTS_7, rng=rng7_std)

print("\n=== ActionPrior HG ===")
rng7_act  = np.random.RandomState(7)
act7      = HyperbolicWorldModelActionPrior(obs_dim=64, latent_dim=16).to(DEVICE)
hist_act7 = train(act7, env7, steps=STEPS_7, beta=1.0,
                  use_actions=True, checkpoint_steps=CKPTS_7, rng=rng7_act)

print("\n--- Final results ---")
print(f"Standard   rho@50k: {hist_std7[-1]['rho']:+.3f}")
print(f"ActionPrior rho@50k: {hist_act7[-1]['rho']:+.3f}")
print(f"Learned depth_bias : {float(F.softplus(act7.log_depth_bias).item()):.4f}")
print(f"Learned shallow_bias: {float(F.softplus(act7.log_shallow_bias).item()):.4f}")


## Experiment 8 — Dimensional efficiency: H^d vs R^d

**Sarkar (2011):** any weighted tree with n nodes embeds in H^2 with distortion → 1
as precision → ∞. In the learning setting, we ask: does H^2 (2-dimensional hyperbolic)
match or beat R^32 (32-dimensional Euclidean) on depth alignment with 15k training steps?

We sweep H^d for d ∈ {2, 4, 8, 16, 32} and R^d for d ∈ {4, 8, 16, 32, 64}.


In [ ]:
STEPS_8  = int(15_000 * STEP_SCALE)
print(f"Exp 8: {STEPS_8} steps per model")
env8     = BaryTreeMDP(B=4, L=5, obs_dim=64, seed=42)
HYP_DIMS = [2, 4, 8, 16, 32]
EUC_DIMS = [4, 8, 16, 32, 64]

results_hyp = {}
for d in HYP_DIMS:
    rng8 = np.random.RandomState(8)
    m    = HyperbolicWorldModel(obs_dim=64, latent_dim=d).to(DEVICE)
    hist = train(m, env8, steps=STEPS_8, beta=1.0, use_actions=False, rng=rng8)
    results_hyp[d] = hist[0]
    print(f"H^{d:2d}  rho={hist[0]['rho']:+.3f}  recon={hist[0]['recon']:.4f}")

print()
results_euc = {}
for d in EUC_DIMS:
    rng8 = np.random.RandomState(8)
    m    = EuclideanWorldModel(obs_dim=64, latent_dim=d).to(DEVICE)
    hist = train(m, env8, steps=STEPS_8, beta=1.0, use_actions=False, rng=rng8)
    results_euc[d] = hist[0]
    print(f"R^{d:2d}  rho={hist[0]['rho']:+.3f}  recon={hist[0]['recon']:.4f}")

print("\n--- Summary ---")
print("dim  H^d rho   R^d rho")
for hd, ed in zip(HYP_DIMS, EUC_DIMS):
    print(f"  {hd:2d}   {results_hyp[hd]['rho']:+.3f}   {results_euc[ed]['rho']:+.3f}  (vs R^{ed})")


## Experiment 9 — OOD depth generalisation

**Setup:** train on `BaryTreeMDP(B=4, L=7)` with walks restricted to depth ≤ 5.
Test on the **same trained model** evaluated on depth-6 and depth-7 nodes (never seen
during training).

**Hypothesis:** the KL geometry enforces a monotone τ-depth ordering.
Since this is a structural property of the prior (not a data-driven correlation),
it should extrapolate: τ at depth 6 should exceed τ at depth 5 without any depth-6
examples in training. The action-conditioned prior makes this extrapolation explicit —
`δ(+1)` biases the prior upward at every depth transition, including unseen ones.


In [ ]:
STEPS_9   = int(20_000 * STEP_SCALE)
MAX_TRAIN = 5
print(f"Exp 9: {STEPS_9} steps")
env9      = BaryTreeMDP(B=4, L=7, obs_dim=64, seed=9)
print(f"Tree: B=4, L=7, n_nodes={env9.n_nodes}")
print(f"Training on depth <= {MAX_TRAIN};  OOD = depth 6-7")

# Standard HG
print("\n=== Standard HG ===")
rng9_std = np.random.RandomState(9)
std9     = HyperbolicWorldModel(obs_dim=64, latent_dim=16).to(DEVICE)
train(std9, env9, steps=STEPS_9, beta=1.0,
      use_actions=False, max_depth=MAX_TRAIN, rng=rng9_std)

rho_std_in  = rho_on_depth_range(std9, env9, rng9_std, 0, MAX_TRAIN, use_actions=False)
rho_std_ood = rho_on_depth_range(std9, env9, rng9_std, MAX_TRAIN+1, env9.L, use_actions=False)
print(f"  rho in-dist (d<=5): {rho_std_in:+.3f}")
print(f"  rho OOD    (d>=6): {rho_std_ood:+.3f}")

# ActionPrior HG
print("\n=== ActionPrior HG ===")
rng9_act = np.random.RandomState(9)
act9     = HyperbolicWorldModelActionPrior(obs_dim=64, latent_dim=16).to(DEVICE)
train(act9, env9, steps=STEPS_9, beta=1.0,
      use_actions=True, max_depth=MAX_TRAIN, rng=rng9_act)

rho_act_in  = rho_on_depth_range(act9, env9, rng9_act, 0, MAX_TRAIN, use_actions=True)
rho_act_ood = rho_on_depth_range(act9, env9, rng9_act, MAX_TRAIN+1, env9.L, use_actions=True)
print(f"  rho in-dist (d<=5): {rho_act_in:+.3f}")
print(f"  rho OOD    (d>=6): {rho_act_ood:+.3f}")

print("\n--- Summary ---")
print(f"Standard   in/OOD: {rho_std_in:+.3f} / {rho_std_ood:+.3f}")
print(f"ActPrior   in/OOD: {rho_act_in:+.3f} / {rho_act_ood:+.3f}")


## Export results to `article_snippets_v2.txt`

In [ ]:
out = {
    "exp7": {
        "standard":     hist_std7,
        "action_prior": hist_act7,
        "depth_bias":   float(F.softplus(act7.log_depth_bias).item()),
        "shallow_bias": float(F.softplus(act7.log_shallow_bias).item()),
    },
    "exp8": {
        "hyperbolic": {str(k): v for k,v in results_hyp.items()},
        "euclidean":  {str(k): v for k,v in results_euc.items()},
    },
    "exp9": {
        "max_train_depth": MAX_TRAIN,
        "standard":     {"rho_in": rho_std_in, "rho_ood": rho_std_ood},
        "action_prior": {"rho_in": rho_act_in, "rho_ood": rho_act_ood},
    },
}

raw = json.dumps(out, indent=2, default=str)
raw = re.sub(r"\bNaN\b", "null", raw)   # safety: Python NaN -> JSON null

with open("article_snippets_v2.txt", "w") as f:
    f.write(raw)

print("Saved article_snippets_v2.txt")
print(raw[:3000])
